# CodeAlpha Data Analytics Internship
## Task 1: Web Scraping & Exploratory Data Analysis

- **Domain**: Data Analytics  
- **Target Site**: [Books to Scrape](http://books.toscrape.com/)  
- **Libraries**: BeautifulSoup4, Requests, Pandas, Matplotlib, Seaborn  

---
### Project Objective
This notebook demonstrates how to:
1. Traverse multi-page e-commerce HTML catalogs.
2. Extract structured fields (Title, Price, Star Rating, Stock, Categories, URLs) using BeautifulSoup.
3. Clean and engineer features with Pandas (handling currency, rating mapping, stock numbers).
4. Conduct Exploratory Data Analysis (EDA) and generate publication-quality visual insights.

In [ ]:
# 1. Import Required Libraries
import os
import re
import time
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set visual style
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print('All libraries imported successfully!')

---
### 2. Inspecting HTML Structure
Before writing an automated scraper, we inspect a sample page to locate CSS classes and HTML hierarchy.

In [ ]:
sample_url = 'http://books.toscrape.com/catalogue/page-1.html'
response = requests.get(sample_url)
soup = BeautifulSoup(response.text, 'html.parser')

sample_card = soup.find('article', class_='product_pod')
title = sample_card.h3.a.get('title', sample_card.h3.a.text)
price = sample_card.find('p', class_='price_color').text.strip()
rating_cls = [c for c in sample_card.find('p', class_='star-rating')['class'] if c != 'star-rating'][0]
stock = sample_card.find('p', class_='instock availability').text.strip()

print(f'Extracted Sample Book:')
print(f'Title:        {title}')
print(f'Price:        {price}')
print(f'Star Rating:  {rating_cls}')
print(f'Availability: {stock}')

---
### 3. Multi-Page Catalog Scraping
Here we scrape multiple catalogue pages systematically.

In [ ]:
def scrape_catalogue(pages=3):
    books = []
    session = requests.Session()
    session.headers.update({'User-Agent': 'Mozilla/5.0 (CodeAlpha Data Analytics Project)'})
    
    for page_num in range(1, pages + 1):
        url = f'http://books.toscrape.com/catalogue/page-{page_num}.html'
        resp = session.get(url)
        if resp.status_code != 200:
            break
        page_soup = BeautifulSoup(resp.text, 'html.parser')
        pods = page_soup.find_all('article', class_='product_pod')
        
        for pod in pods:
            a_tag = pod.h3.find('a')
            b_title = a_tag.get('title', a_tag.text.strip())
            b_url = urljoin(url, a_tag['href'])
            b_price = pod.find('p', class_='price_color').text.strip()
            r_classes = pod.find('p', class_='star-rating')['class']
            b_rating = [c for c in r_classes if c != 'star-rating'][0]
            b_avail = pod.find('p', class_='instock availability').text.strip()
            
            books.append({
                'title': b_title,
                'price_raw': b_price,
                'rating_str': b_rating,
                'availability_raw': b_avail,
                'book_url': b_url,
                'page': page_num
            })
    return pd.DataFrame(books)

df_raw = scrape_catalogue(pages=3)
print(f'Scraped {len(df_raw)} books from 3 pages.')
df_raw.head()

---
### 4. Data Cleaning & Feature Engineering
We clean prices, parse numeric ratings, and extract availability numbers.

In [ ]:
RATING_MAP = {'One': 1, 'Two': 2, 'Three': 3, 'Four': 4, 'Five': 5}

df = df_raw.copy()
df['price_gbp'] = df['price_raw'].astype(str).str.replace(r'[^0-9.]', '', regex=True).astype(float)
df['rating_num'] = df['rating_str'].map(RATING_MAP).fillna(0).astype(int)
df['in_stock'] = df['availability_raw'].str.contains('In stock', case=False)
df['price_tier'] = pd.cut(
    df['price_gbp'],
    bins=[0, 20, 40, float('inf')],
    labels=['Budget (<£20)', 'Mid-Range (£20-£40)', 'Premium (>£40)']
)

print('Cleaned dataset info:')
df.info()
df[['title', 'price_gbp', 'rating_num', 'in_stock', 'price_tier']].head()

---
### 5. Exploratory Data Analysis & Visualizations

In [ ]:
# 5.1 Price Distribution
plt.figure(figsize=(9, 4.5))
sns.histplot(df['price_gbp'], kde=True, color='#2563eb', bins=15, edgecolor='white')
m_val = df['price_gbp'].mean()
plt.axvline(m_val, color='#dc2626', linestyle='--', label=f'Mean: £{m_val:.2f}')
plt.title('Distribution of Book Prices (£)')
plt.xlabel('Price in GBP (£)')
plt.ylabel('Frequency')
plt.legend()
plt.show()

In [ ]:
# 5.2 Ratings Breakdown
plt.figure(figsize=(8, 4.5))
r_counts = df['rating_num'].value_counts().sort_index()
r_labels = [f'{i} Stars' for i in r_counts.index]
sns.barplot(x=r_labels, y=r_counts.values, hue=r_labels, legend=False, palette='Blues_d')
plt.title('Star Ratings Frequency')
plt.xlabel('Rating')
plt.ylabel('Count')
plt.show()

---
### 6. Key Findings & Conclusion
- The scraping pipeline successfully acquired and structured real-time e-commerce book data.
- Data cleaning resolved unstructured text ratings, currency characters, and categorical bucketing.
- Visualizations indicate broad pricing variation across titles and balanced ratings distributions.
- Launch `streamlit run app.py` for full interactive exploration!